# 01 · Chunking strategies and semantic chunking

After this notebook you can cut documents into retrieval-ready chunks five different ways (fixed-size with overlap, recursive, sentence, heading-aware and semantic), explain what each one breaks, and prove with a measured experiment which chunk size your eval set prefers.

**Time:** ~30 min · **Runs:** offline on CPU in ~1 min (one 90 MB embedding model, cached) · **Needs:** [embeddings and similarity](../07_genai_foundations/02_embeddings_and_similarity.ipynb)

## Why this matters in interviews

- Chunking is the **single biggest quality lever** in RAG and the first thing a senior interviewer probes: "why not index whole documents?", "what chunk size, and how did you choose it?".
- The strong answer is not an opinion but an **experiment**: sweep the size, measure hit-rate on labelled questions, look at the cost. You will run exactly that here.
- Follow-ups you should be able to code on the spot: overlap, heading-aware splitting with the heading path as metadata, semantic chunking with a percentile threshold, and parent-child (small-to-big) retrieval.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `rg03` | What are chunks, and why not just index whole documents? |
| `rg04` | What are the six types of chunking, and when do you use each? |
| `rg41` | What is document-aware chunking and why is the heading path so valuable? |
| `rg42` | What is semantic chunking and is it worth the cost? |
| `rg05` | What is parent-child (small-to-big) retrieval? |
| `rg44` | What goes in a chunk's metadata, and why is it not optional? |
| `rg06` | What is contextual retrieval and why does it work so well? (and late chunking) |

## Setup: the knowledge base, the questions, the embedder

Every notebook in this section uses the same tiny corpus: 15 markdown docs from a fictional company, **Acme Cloud** (refunds, shipping, warranty, API limits and error codes, on-call, pricing, SSO, data retention, ...). The next cell is pure data - skim it once.

In [ ]:
import hashlib
import json
import re
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

random.seed(0)
np.random.seed(0)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 70)

In [ ]:
# The knowledge base: 15 short markdown docs for a fictional company, Acme Cloud (a SaaS analytics product
# that also sells an 'Edge Gateway' device). This cell is data - it is identical in every 09_rag notebook.
KB = {
"refund_policy.md": """# Refund Policy

## Eligibility
Annual plans can be refunded in full within 30 days of purchase. Monthly plans can be cancelled at any time, but the current month is not refunded. Add-on credits and professional-services hours are non-refundable.

## How to request a refund
Open the billing portal, choose the invoice and click "Request refund". Only workspace owners can request refunds.

## Processing time
Approved refunds go back to the original payment method within 5-10 business days. Bank transfers can take up to 15 business days.
""",
"shipping.md": """# Hardware Shipping

## Delivery times
Edge Gateway devices ship from our warehouses in Rotterdam and Dallas. Standard delivery takes 3-5 business days. Express delivery takes 1-2 business days and costs $25.

## Tracking
A tracking link is emailed when the order leaves the warehouse. The first carrier scan can take 24 hours to appear.

## Lost or damaged shipments
If a shipment has not arrived 10 business days after dispatch, report it within 14 days so we can open a carrier investigation. We send a free replacement when the carrier confirms the loss.
""",
"warranty.md": """# Edge Gateway Warranty

## Coverage
Every Edge Gateway has a 2-year limited hardware warranty from the delivery date. It covers manufacturing defects in the device, the power supply and the antennas.

## Not covered
Water damage, drops, unauthorised repairs and cosmetic scratches are not covered by the warranty.

## RMA process
Request an RMA (return merchandise authorisation) in the support portal with the device serial number. Once the RMA is approved we ship a replacement within 7 days, and you send the faulty unit back with the prepaid label.
""",
"api_rate_limits.md": """# API Rate Limits

## Limits per plan
Free: 60 requests per minute. Pro: 600 requests per minute. Business: 1,200 requests per minute. Enterprise: 6,000 requests per minute. Limits apply per API key.

## Best practices
Retry with exponential backoff and jitter, cache responses you read often, and send writes through the bulk endpoint.
""",
"api_errors.md": """# API Error Reference

## ERR-429-A
The per-minute rate limit was exceeded. Wait for the number of seconds in the Retry-After header, then retry.

## ERR-429-B
The daily request quota is exhausted. The quota resets at 00:00 UTC; upgrade the plan for a higher quota.

## ERR-503-C
The API is in scheduled maintenance. Retry after the maintenance window shown on the status page.

## ERR-401-D
The API key was revoked or has expired. Create a new key in Settings > API keys.
""",
"gateway_models.md": """# Edge Gateway Models

## EG-100
The EG-100 is the entry model: 2 Ethernet ports, Wi-Fi 5 and an 8 W maximum power draw. It supports up to 50 sensors.

## EG-200
The EG-200 adds LTE backup: 4 Ethernet ports, Wi-Fi 6 and a 12 W maximum power draw. It supports up to 200 sensors.

## EG-300
The EG-300 is the rugged outdoor model: an IP67 enclosure, 4 Ethernet ports, Wi-Fi 6 and an 18 W maximum power draw. It supports up to 500 sensors.
""",
"oncall_policy.md": """# On-call Policy

## Rotation
Engineers join the on-call rotation after their third month. Each shift lasts one week and hands over on Monday at 09:00 local time. Every shift has a primary and a secondary.

## Response targets
A SEV1 page must be acknowledged within 5 minutes and a SEV2 page within 30 minutes. If a SEV1 page is not acknowledged within 15 minutes, it escalates to the secondary and then to the engineering manager.

## Compensation
On-call engineers receive a stipend of $300 per week, and a day off after any night with more than two pages.
""",
"pricing.md": """# Pricing Plans

## Plans
Free costs $0 and includes 3 users and 1 GB of storage. Pro costs $49 per user per month. Business costs $99 per user per month. Enterprise pricing is custom and starts at 50 users.

## What each plan includes
Single sign-on (SAML and OIDC) is included from the Business plan upward. Audit logs and the 99.95% uptime SLA are included only in Enterprise. All paid plans include email support.

## Billing
Paying annually saves 20% compared with monthly billing. Prices exclude VAT.
""",
"sso_setup.md": """# Single Sign-On Setup

## Supported identity providers
Acme Cloud supports Okta, Microsoft Entra ID (formerly Azure AD) and Google Workspace through SAML 2.0 or OIDC.

## Configuration steps
Go to Settings > Security > Single sign-on, upload the identity provider's metadata XML, and map the email and groups attributes. Test with one account before you enforce SSO for the whole workspace.

## Troubleshooting
SSO-ERR-12 means the SAML assertion was rejected because the clocks differ by more than 5 minutes; sync the identity provider's clock with NTP. SSO-ERR-31 means the email attribute is missing from the assertion.
""",
"data_retention.md": """# Data Retention

## Event data
Raw event data is kept for 30 days on Free, 13 months on Pro and Business, and 25 months on Enterprise. Aggregated reports are kept for as long as the workspace exists.

## Deleted workspaces
When a workspace is deleted, its data is purged from primary storage after 30 days and from backups after a further 35 days.

## Erasure requests
GDPR erasure requests for one person's data are completed within 30 days of identity verification.
""",
"pto_policy.md": """# Paid Time Off

## Allowance
Full-time employees receive 25 days of paid time off per calendar year, plus public holidays. Part-time employees receive a pro-rata allowance.

## Booking
Book absences longer than three days in Workday at least two weeks ahead. Your manager approves the request.

## Carry-over
Up to 5 unused days can be carried into the next year and must be used by 31 March.
""",
"webhooks.md": """# Webhooks

## Delivery and retries
Webhook events are sent as HTTPS POST requests. If your endpoint does not return a 2xx status within 10 seconds, we retry up to 8 times with exponential backoff over 24 hours.

## Verifying signatures
Every request carries an X-Acme-Signature header: an HMAC-SHA256 of the raw body computed with your webhook secret. Compare it in constant time and reject requests older than 5 minutes.
""",
"sla.md": """# Service Level Agreement

## Uptime commitment
Pro and Business are backed by a 99.9% monthly uptime commitment. Enterprise is backed by 99.95%.

## Service credits
If monthly uptime falls below the commitment, you receive a service credit of 10% of the monthly fee; below 99.0% the credit is 25%. Credits are applied to the next invoice.

## How to claim
Open a support ticket within 30 days of the end of the affected month and include the incident dates.
""",
"account_security.md": """# Account Security

## Passwords
Password reset links expire after 60 minutes. After 5 failed sign-in attempts the account is locked for 15 minutes.

## Two-factor authentication
Workspace owners can require two-factor authentication (2FA) for every member. Authenticator apps and hardware security keys are supported; SMS codes are not.
""",
"expenses.md": """# Travel and Expenses

## Meals and hotels
Meals are reimbursed up to $75 per day while travelling. Book hotels through the travel portal; the nightly cap is $200 in most cities.

## Submitting claims
Submit receipts in Workday within 30 days of the expense. Claims over $25 without a receipt are rejected.
""",
}

The eval set below is what makes chunking measurable. Each question carries **evidence**: the doc and a phrase the retrieved chunk must contain. "Did retrieval return the evidence?" is a question with a yes/no answer for *any* chunking scheme - that is what lets us compare schemes fairly. (A real project uses 50-100 real questions; 14 is enough to see the mechanics.)

In [ ]:
# The eval set: each question lists its EVIDENCE - (doc, phrase) pairs that a retrieved chunk must contain.
# Phrases are unique in the KB, so "is the evidence in the retrieved text?" works for any chunking scheme.
QUESTIONS = [
    {"q": "What does ERR-429-A mean?", "kind": "exact id", "evidence": [("api_errors.md", "per-minute rate limit")]},
    {"q": "Our integration keeps failing with ERR-401-D", "kind": "exact id", "evidence": [("api_errors.md", "revoked")]},
    {"q": "How do I fix SSO-ERR-12?", "kind": "exact id", "evidence": [("sso_setup.md", "clocks differ")]},
    {"q": "How many sensors does the EG-300 support?", "kind": "exact id", "evidence": [("gateway_models.md", "500 sensors")]},
    {"q": "How do I get my money back after buying an annual plan by mistake?", "kind": "paraphrase",
     "evidence": [("refund_policy.md", "refunded in full within 30 days")]},
    {"q": "My parcel never showed up. What should I do?", "kind": "paraphrase", "evidence": [("shipping.md", "report it within 14 days")]},
    {"q": "How many vacation days do I get a year?", "kind": "paraphrase", "evidence": [("pto_policy.md", "25 days of paid time off")]},
    {"q": "Do we get compensated for outages?", "kind": "paraphrase", "evidence": [("sla.md", "service credit")]},
    {"q": "Which gateway can be installed outdoors?", "kind": "paraphrase", "evidence": [("gateway_models.md", "rugged outdoor model")]},
    {"q": "Who gets paged when nobody acknowledges a SEV1 alert?", "kind": "keyword", "evidence": [("oncall_policy.md", "engineering manager")]},
    {"q": "How long is event data kept on the Pro plan?", "kind": "keyword", "evidence": [("data_retention.md", "13 months on Pro")]},
    {"q": "What do we get back if the platform is down more than promised?", "kind": "vague", "evidence": [("sla.md", "service credit")]},
    {"q": "What is the API rate limit on the cheapest plan that includes SSO?", "kind": "multi-hop",
     "evidence": [("pricing.md", "from the Business plan upward"), ("api_rate_limits.md", "Business: 1,200 requests")]},
    {"q": "How long do approved refunds take, and how long is the gateway warranty?", "kind": "multi-part",
     "evidence": [("refund_policy.md", "5-10 business days"), ("warranty.md", "2-year")]},
    {"q": "What is the capital of Australia?", "kind": "out-of-scope", "evidence": []},
]
IN_SCOPE = [q for q in QUESTIONS if q["evidence"]]


def has_evidence(text, doc, q):
    """True if this text (from `doc`) contains any of the question's evidence phrases."""
    return any(doc == d and p.lower() in text.lower() for d, p in q["evidence"])


print(f"{len(KB)} docs, {sum(len(t) for t in KB.values()):,} characters; "
      f"{len(QUESTIONS)} questions ({len(IN_SCOPE)} answerable, {len(QUESTIONS) - len(IN_SCOPE)} out of scope)")

In [ ]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import transformers
transformers.utils.logging.disable_progress_bar()          # keep the cell output clean
transformers.logging.set_verbosity_error()
from sentence_transformers import SentenceTransformer


def load_st(name, cls=SentenceTransformer):
    """Load a small Hugging Face model: cached copy first (no network), download only if missing."""
    try:
        return cls(name, local_files_only=True)
    except Exception:
        try:
            return cls(name)                                 # first run only: ~90 MB download
        except Exception as e:
            raise RuntimeError(f"Could not load {name}: connect to the internet once so it can download.") from e


embedder = load_st("sentence-transformers/all-MiniLM-L6-v2")


def encode(texts):
    """Unit-length MiniLM embeddings (384-d), so a dot product IS the cosine similarity."""
    return embedder.encode(list(texts), normalize_embeddings=True, convert_to_numpy=True)


print("embedder: all-MiniLM-L6-v2 |", embedder.get_embedding_dimension(), "dims | max",
      embedder.max_seq_length, "tokens per text (the rest is silently cut off)")

## 1. Why chunk at all?

**In plain English:** a whole document is *about* several things, so its single embedding is an average of all of them and sits close to none. A section is about one thing. Two effects, both measurable:

1. **Dilution** - the question matches its section better than the whole document.
2. **Truncation** - the embedder only reads its first `max_seq_length` tokens (256 for MiniLM). Anything after that is invisible, silently.

In [ ]:
question = "How long do approved refunds take to arrive?"
doc = KB["refund_policy.md"]
sections = [s.strip() for s in doc.split("\n## ")[1:]]

q_vec = encode([question])[0]
whole_vec = encode([doc])[0]
sec_vecs = encode(sections)

labels = ["whole doc"] + [s.splitlines()[0] for s in sections]
scores = [float(q_vec @ whole_vec)] + [float(q_vec @ v) for v in sec_vecs]
for lab, s in zip(labels, scores):
    print(f"cos(question, {lab:22s}) = {s:.3f}")

best_section = labels[1 + int(np.argmax(scores[1:]))]
assert best_section == "Processing time" and max(scores[1:]) > scores[0], "the answering section should beat the whole doc"

plt.figure(figsize=(7, 2.8))
plt.barh(labels[::-1], scores[::-1], color=["#3b5bdb" if lab == best_section else "#adb5bd" for lab in labels[::-1]])
plt.xlabel("cosine similarity to the question")
plt.title("The answering section beats the whole document (dilution)")
plt.tight_layout()
plt.show()

The "Processing time" section scores well above the whole document, even though the whole document *contains* that section - the other two sections drag its average away. Now truncation:

In [ ]:
long_text = (doc + "\n") * 4                                       # ~4x the refund policy, well over 256 tokens
n_tokens = len(embedder.tokenizer(long_text)["input_ids"])
tail = " PS: the warranty on penguins is void in Antarctica." * 20   # appended AFTER the 256-token window

v1, v2 = encode([long_text, long_text + tail])
print(f"tokens in long_text: {n_tokens} (window = {embedder.max_seq_length})")
print(f"cosine(long_text, long_text + unrelated tail) = {float(v1 @ v2):.6f}")
assert n_tokens > embedder.max_seq_length
assert np.allclose(v1, v2, atol=1e-5), "text past the window must not change the embedding"

Identical embeddings: the penguin paragraph does not exist as far as retrieval is concerned. If you embed 5-page documents with a 256-token model, **everything after roughly the first page is unsearchable** and nothing errors. Always check the embedder's max length against your chunk size.

## 2. Fixed-size chunks with overlap, from scratch

The simplest scheme: cut every `size` characters, and start each chunk `size - overlap` after the previous one so a sentence that straddles a cut appears whole in at least one chunk. Write it, then *prove* it covers the text.

In [ ]:
def fixed_size_chunks(text, size=120, overlap=30):
    """Return (start, chunk) pairs. Consecutive chunks share `overlap` characters."""
    assert 0 <= overlap < size, "overlap must be smaller than size"
    step = size - overlap
    out = []
    for start in range(0, len(text), step):
        out.append((start, text[start:start + size]))
        if start + size >= len(text):
            break
    return out


text = KB["refund_policy.md"]
fixed = fixed_size_chunks(text, size=120, overlap=30)

covered = np.zeros(len(text), dtype=bool)
for start, chunk in fixed:
    assert text[start:start + len(chunk)] == chunk
    covered[start:start + len(chunk)] = True
assert covered.all(), "every character must land in at least one chunk"
for (s1, c1), (s2, c2) in zip(fixed, fixed[1:]):
    assert c1[s2 - s1:] == c2[: len(c1) - (s2 - s1)], "neighbours must share the overlap"
rebuilt = fixed[0][1] + "".join(c[30:] for _, c in fixed[1:])
assert rebuilt == text, "dropping each overlap must give back the original text"

print(f"{len(text)} chars -> {len(fixed)} chunks; coverage, overlap and reconstruction all verified\n")
for start, chunk in fixed[:4]:
    print(f"[{start:3d}] {chunk!r}")

It is correct - and ugly. Chunks start and end in the middle of words ("...refunded in fu" / "ll within..."). Count how often that happens; then compare with a splitter that knows about paragraphs and sentences.

In [ ]:
def cuts_mid_word(chunks_text, full_text):
    """How many chunk boundaries fall inside a word."""
    bad = 0
    for chunk in chunks_text:
        start = full_text.find(chunk)
        end = start + len(chunk)
        if start > 0 and full_text[start - 1].isalnum() and full_text[start].isalnum():
            bad += 1
        if end < len(full_text) and full_text[end - 1].isalnum() and full_text[end].isalnum():
            bad += 1
    return bad


fixed_texts = [c for _, c in fixed]
print(f"fixed-size: {len(fixed_texts)} chunks, {cuts_mid_word(fixed_texts, text)} boundaries inside a word")

## 3. Recursive splitting (`RecursiveCharacterTextSplitter`)

**In plain English:** try to cut at paragraph breaks; if a piece is still too big, cut at line breaks, then spaces, and only as a last resort mid-word. It is LangChain's default and the right day-one choice. Same size budget as above:

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=30, add_start_index=True)
rec_docs = recursive.create_documents([text], metadatas=[{"source": "refund_policy.md"}])
rec_texts = [d.page_content for d in rec_docs]

print("separators tried, in order:", recursive._separators)
print(f"recursive : {len(rec_texts)} chunks, {cuts_mid_word(rec_texts, text)} boundaries inside a word\n")
for d in rec_docs[:5]:
    print(f"[{d.metadata['start_index']:3d}] {d.page_content!r}")
assert cuts_mid_word(rec_texts, text) < cuts_mid_word(fixed_texts, text)

Zero (or almost zero) mid-word cuts, and a lone heading like `'## Eligibility'` shows the other weakness: the recursive splitter is blind to *meaning* - it happily emits a heading as its own useless chunk, separated from the text it introduces. The next two schemes fix that from different angles.

## 4. Sentence-based chunks

Split into sentences, then pack whole sentences greedily until the size budget is reached. No sentence is ever cut.

In [ ]:
def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n{2,}", text) if s.strip()]


def sentence_chunks(text, max_chars=160):
    body = "\n".join(line for line in text.splitlines() if not line.startswith("#"))   # headings are not sentences
    chunks, current = [], ""
    for sent in split_sentences(body):
        if current and len(current) + 1 + len(sent) > max_chars:
            chunks.append(current)
            current = sent
        else:
            current = f"{current} {sent}".strip()
    if current:
        chunks.append(current)
    return chunks


sent_texts = sentence_chunks(text)
for c in sent_texts:
    print(repr(c))
body_sents = split_sentences(" ".join(line for line in text.splitlines() if not line.startswith("#")))
assert all(any(s in c for c in sent_texts) for s in body_sents), "every sentence survives intact in one chunk"

## 5. Heading-aware chunks (`MarkdownHeaderTextSplitter`) - keep the heading path

Most company knowledge (wikis, manuals, policies) already has structure. Split on the headings, **store the heading path as metadata, and prepend it to the chunk text**. A chunk that says "5-10 business days" now also says "Refund Policy > Processing time" - exactly the words a question uses.

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "doc_title"), ("##", "section")],
                                         strip_headers=True)
md_docs = md_splitter.split_text(KB["refund_policy.md"])
for d in md_docs:
    print(d.metadata, "->", d.page_content[:60] + "...")

# Prepend the heading path, and fall back to recursive splitting for any section that is still too long.
long_section_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60)
kb_chunks = []
for name, body in KB.items():
    for d in long_section_splitter.split_documents(md_splitter.split_text(body)):
        path = f"{d.metadata.get('doc_title', '')} > {d.metadata.get('section', '')}"
        kb_chunks.append({"source": name, "section_path": path, "text": f"{path}\n{d.page_content}"})

print(f"\nwhole KB -> {len(kb_chunks)} heading-aware chunks; example:\n{kb_chunks[2]['text']}")
assert all(c["text"].startswith(c["section_path"]) for c in kb_chunks)

## 6. Semantic chunking: cut where the topic changes

For text with **no structure** (meeting notes, transcripts, long emails) there are no headings to split on. Semantic chunking finds the boundaries from meaning:

1. split into sentences and embed each one;
2. compute the cosine **distance** between each sentence and the next;
3. a big jump means a topic change: cut wherever the distance is above a **percentile threshold** of all the distances.

The notes below cover four topics; we know where the true boundaries are, so we can check the algorithm.

In [ ]:
NOTES = ("Mamitha opened the meeting with the refund backlog. Forty refund requests have been waiting for more than ten business days. "
         "Most of them are bank transfers, which the payment provider processes slowly. Finance will send a weekly report of the refunds that are still pending. "
         "Next, Tom raised the on-call load. The primary engineer was paged eleven times on Saturday night. "
         "Half of those pages came from one flaky disk alert. The team will tune that alert and add a third person to the rotation. "
         "Marketing wants to launch the EG-300 outdoor gateway in May. The EG-300 still needs its final IP67 certificate from the test lab. "
         "The first five hundred EG-300 units will ship from the Rotterdam warehouse. "
         "Finally, a large customer reported single sign-on failures after switching identity providers. "
         "Their SAML assertions were rejected because of clock drift. Support will publish a short guide on syncing clocks with NTP.")
TRUE_STARTS = [4, 8, 11]          # sentence index where topics 2, 3 and 4 begin

sents = split_sentences(NOTES)
sent_vecs = encode(sents)
distances = 1 - np.sum(sent_vecs[:-1] * sent_vecs[1:], axis=1)     # distance between sentence i and i+1


def semantic_breakpoints(distances, percentile):
    threshold = np.percentile(distances, percentile)
    return threshold, [i + 1 for i, d in enumerate(distances) if d > threshold]   # new chunk starts at i+1


threshold, starts = semantic_breakpoints(distances, 80)
semantic_chunks = [" ".join(sents[a:b]) for a, b in zip([0] + starts, starts + [len(sents)])]
print(f"{len(sents)} sentences, threshold (80th percentile) = {threshold:.3f}, new chunks start at {starts}")
for c in semantic_chunks:
    print(" -", c[:95] + "...")
assert starts == TRUE_STARTS, "at the 80th percentile the cuts land exactly on the topic changes"

In [ ]:
plt.figure(figsize=(10, 3.8))
x = np.arange(1, len(sents))
plt.plot(x, distances, "o-", color="#3b5bdb", label="distance to previous sentence")
plt.axhline(threshold, color="#e8590c", ls="--", label="80th-percentile threshold")
for s in TRUE_STARTS:
    plt.axvspan(s - 0.3, s + 0.3, color="#b2f2bb", alpha=0.6)
for s in starts:
    plt.annotate("cut", (s, distances[s - 1]), textcoords="offset points", xytext=(0, 8), ha="center", color="#e8590c")
plt.xlabel("sentence index (green band = true topic change)")
plt.ylabel("cosine distance")
plt.title("Semantic chunking: cut where the neighbour distance spikes")
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

Look at the margins: the within-topic jump at sentence 7 is almost as high as the real boundary at 11. The threshold is a **hyper-parameter**, and the default in libraries (the 95th percentile) assumes documents with hundreds of sentences. Sweep it:

In [ ]:
rows = []
for pct in [60, 70, 80, 90, 95]:
    thr, st = semantic_breakpoints(distances, pct)
    rows.append({"percentile": pct, "threshold": round(thr, 3), "chunks": len(st) + 1, "cuts at": st,
                 "true cuts found": len(set(st) & set(TRUE_STARTS)), "false cuts": len(set(st) - set(TRUE_STARTS))})
print(pd.DataFrame(rows).to_string(index=False))

Low percentiles over-split (false cuts inside a topic), high ones under-split (topics glued together). That is the honest cost of semantic chunking: **one embedding per sentence at index time, plus a threshold you must tune per corpus**. The library version is `SemanticChunker` in `langchain_experimental` (not installed here; same algorithm):

```python
from langchain_experimental.text_splitter import SemanticChunker   # pip install langchain-experimental
splitter = SemanticChunker(embeddings, breakpoint_threshold_type="percentile", breakpoint_threshold_amount=95)
docs = splitter.create_documents([long_text])
```

It also combines each sentence with its neighbours before embedding (`buffer_size=1`) to smooth the distances. **Rule of thumb:** if the corpus has headings, heading-aware chunking beats semantic chunking and costs nothing.

## 7. The trade-offs in one table

| Scheme | Cuts on | Index cost | Respects meaning | Best for | Main failure |
|---|---|---|---|---|---|
| Fixed size | N characters/tokens | lowest | no | logs, transcripts | cuts words and sentences in half |
| Fixed + overlap | N with a stride | low | barely | a safe baseline | duplicates content (redundant top-k) |
| Recursive | paragraph → line → word | low | punctuation only | general prose; the day-one default | blind to topic; orphan headings |
| Sentence packing | sentence ends | low | sentences | FAQs, short answers | still blind to topic |
| Heading-aware | headings / DOM / AST | medium (needs a parser) | structure | manuals, policies, wikis, code | one huge section still needs a second split |
| Semantic | embedding distance | high (embed every sentence) | yes | unstructured mixed-topic text | threshold tuning per corpus |
| Parent-child | small children, big parents | medium (two stores) | via the parent | almost everything | parents can blow the context budget |

**Size trade-off:** small chunks retrieve precisely but may not contain the whole answer; big chunks contain it but embed vaguely, may exceed the embedder's window, and cost more tokens in every prompt. The only way to choose is to measure.

## 8. Experiment: hit-rate vs chunk size

For each chunk size: split the whole KB with the recursive splitter (20% overlap), embed, retrieve the top 3 chunks per question, and count a **hit** if the evidence phrase is in one of them. Also record how many characters we would paste into the prompt (the cost side). The heading-aware section chunks from the next notebooks are plotted as a reference point.

In [ ]:
def evaluate_chunks(chunk_texts, chunk_docs, k=3):
    vecs = encode(chunk_texts)
    q_vecs = encode([q["q"] for q in IN_SCOPE])
    hit1, hitk, ctx = [], [], []
    for qv, q in zip(q_vecs, IN_SCOPE):
        order = np.argsort(-(vecs @ qv))
        rel = [has_evidence(chunk_texts[i], chunk_docs[i], q) for i in order]
        hit1.append(rel[0])
        hitk.append(any(rel[:k]))
        ctx.append(sum(len(chunk_texts[i]) for i in order[:k]))
    return np.mean(hit1), np.mean(hitk), np.mean(ctx)


sweep = []
for size in [50, 80, 120, 160, 200, 300, 450, 700]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=size // 5)
    pairs = [(t, name) for name, body in KB.items() for t in splitter.split_text(body)]
    h1, h3, ctx = evaluate_chunks([t for t, _ in pairs], [d for _, d in pairs])
    sweep.append({"chunk_size": size, "n_chunks": len(pairs), "hit@1": h1, "hit@3": h3, "context_chars@3": ctx})

sec_h1, sec_h3, sec_ctx = evaluate_chunks([c["text"] for c in kb_chunks], [c["source"] for c in kb_chunks])
sweep_df = pd.DataFrame(sweep)
print(sweep_df.round(3).to_string(index=False))
print(f"\nheading-aware sections: {len(kb_chunks)} chunks, hit@1={sec_h1:.3f}, hit@3={sec_h3:.3f}, context_chars@3={sec_ctx:.0f}")
assert sweep_df["hit@3"].iloc[0] < 0.5 < sweep_df["hit@3"].iloc[-1], "tiny chunks should miss most evidence"

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(sweep_df["chunk_size"], sweep_df["hit@3"], "o-", label="hit@3", color="#3b5bdb")
ax1.plot(sweep_df["chunk_size"], sweep_df["hit@1"], "s--", label="hit@1", color="#74c0fc")
ax1.scatter([sec_ctx / 3], [sec_h3], marker="*", s=220, color="#e8590c", zorder=5, label="heading-aware (avg size)")
ax1.set_xscale("log")
ax1.set_xlabel("chunk size (characters, log scale)")
ax1.set_ylabel("fraction of questions")
ax1.set_ylim(0, 1.05)
ax1.set_title("Retrieval quality vs chunk size")
ax1.legend(loc="lower right")
ax2.plot(sweep_df["chunk_size"], sweep_df["context_chars@3"], "o-", color="#868e96")
ax2.set_xscale("log")
ax2.set_xlabel("chunk size (characters, log scale)")
ax2.set_ylabel("characters pasted into the prompt (top 3)")
ax2.set_title("...and what it costs per question")
plt.tight_layout()
plt.show()

How to read it honestly:

- **The left side is the real lesson.** Below ~100 characters the chunks cut the evidence apart or lose the words that match the question, and hit-rate collapses.
- **The right side is flattered by this toy corpus.** Each Acme doc is only ~450 characters, so a "700-character chunk" is a whole document, and top-3 means "a fifth of the corpus". On a real corpus (documents of 5-50 pages) whole-document chunks are impossible: the embedder truncates at 256 tokens (section 1) and top-3 of a million chunks is not generous.
- **The cost grows linearly** (right panel) while quality flattens: past the knee you pay for tokens, latency and lost-in-the-middle, not for recall.
- The **heading-aware star** gets close to the best recursive setting with far less context per question - the cheapest good option when the documents have structure.

In a real project you run this exact loop on 50-100 labelled questions, plus overlap (0/10/20%) and k, and pick the knee.

## 9. Metadata: what every chunk should carry

A chunk is not just text. Without metadata you cannot filter (tenant, permissions), cite (source, section), expire (version, updated_at) or upsert idempotently (content hash).

In [ ]:
def make_record(source, section_path, chunk_index, text, version="2026-09-01"):
    return {
        "chunk_id": f"{source}#{chunk_index}",
        "source_id": source,
        "section_path": section_path,
        "chunk_index": chunk_index,
        "doc_version": version,
        "updated_at": "2026-09-01T09:00:00Z",
        "language": "en",
        "tenant_id": "acme",
        "acl": ["employees"] if source in {"pto_policy.md", "oncall_policy.md", "expenses.md"} else ["everyone"],
        "embedding_model": "all-MiniLM-L6-v2",           # guards against mixing vector spaces (rg14)
        "content_hash": hashlib.sha256(text.encode()).hexdigest()[:16],
        "text": text,
    }


records = [make_record(c["source"], c["section_path"], i, c["text"]) for i, c in enumerate(kb_chunks)]
print(json.dumps({k: v for k, v in records[2].items()}, indent=2)[:700])

The content hash makes re-ingestion **idempotent**: re-embed only chunks whose hash changed. Simulate an index, then edit one sentence of one document:

In [ ]:
index = {r["chunk_id"]: r["content_hash"] for r in records}


def upsert(new_records, index):
    changed = [r for r in new_records if index.get(r["chunk_id"]) != r["content_hash"]]
    for r in changed:
        index[r["chunk_id"]] = r["content_hash"]
    return changed


assert upsert(records, index) == [], "re-ingesting unchanged docs must re-embed nothing"

edited = [dict(c) for c in kb_chunks]
for c in edited:
    if c["source"] == "refund_policy.md" and "15 business days" in c["text"]:
        c["text"] = c["text"].replace("15 business days", "20 business days")
new_records = [make_record(c["source"], c["section_path"], i, c["text"]) for i, c in enumerate(edited)]
changed = upsert(new_records, index)
print("chunks to re-embed after the edit:", [r["chunk_id"] for r in changed])
assert len(changed) == 1

## 10. Parent-child (small-to-big) retrieval

**In plain English:** search with small, precise pieces; hand the model the whole section they came from. Index **children** (single sentences) for precision; each child points to its **parent** (the section). Retrieve children, **deduplicate by parent**, return parents.

In [ ]:
parents = {i: c for i, c in enumerate(kb_chunks)}
children = [(pid, s) for pid, c in parents.items() for s in split_sentences(c["text"].split("\n", 1)[1])]
child_vecs = encode([s for _, s in children])
print(f"{len(parents)} parents, {len(children)} children (sentences)")


def small_to_big(question, k_children=6, max_parents=2):
    order = np.argsort(-(child_vecs @ encode([question])[0]))[:k_children]
    hit_children = [children[i] for i in order]
    parent_ids = list(dict.fromkeys(pid for pid, _ in hit_children))[:max_parents]   # dedupe, keep rank order
    return hit_children, [parents[p] for p in parent_ids]


hit_children, hit_parents = small_to_big("Can I get my money back on a monthly plan?")
print("\ntop children:")
for pid, s in hit_children:
    print(f"  parent {pid:2d} | {s}")
print("\nreturned to the LLM (deduplicated parents):")
for p in hit_parents:
    print("  ---", p["text"].replace("\n", " | "))
assert len(hit_parents) < len(hit_children), "several children share a parent: dedupe saves context"
assert any("Monthly plans" in p["text"] for p in hit_parents)

The child sentence "Monthly plans can be cancelled at any time, but the current month is not refunded" is a precise match; the parent adds the surrounding rules (the 30-day window for annual plans, what is never refundable) that a good answer needs. Without the dedupe step, several children of the same section would paste that section several times.

A cheaper variant, **sentence-window retrieval**, returns the matching sentence plus one or two neighbours instead of the whole section (exercise 3).

## 11. Contextual retrieval and late chunking (briefly)

Both attack the same problem: **a chunk cut out of its document loses its context** ("it takes 5-10 days" - what does?).

- **Heading path** (what we did in section 5): free, when the structure exists.
- **Contextual retrieval** (Anthropic): at index time, ask a cheap LLM to write 1-2 sentences situating each chunk in its document, prepend them before embedding *and* before BM25. Reported ~35% fewer retrieval failures (~49% combined with BM25). Cost: one cheap call per chunk, once, discounted by prompt-caching the document.
- **Late chunking**: embed the *whole document* with a long-context embedding model first, then average the token embeddings inside each chunk's span. Every chunk vector has "seen" the whole document; no LLM call. It needs a long-context embedder (e.g. 8k tokens) - MiniLM's 256-token window cannot do it, which is why it is described here rather than run.

## Try it yourself

**Exercise 1.** Set the overlap to 0 in `fixed_size_chunks` and count how many *sentences* of the refund policy no longer appear intact in any chunk.

In [ ]:
# Solution — try it yourself first, then run this
for overlap in (0, 30, 60):
    pieces = [c for _, c in fixed_size_chunks(text, size=120, overlap=overlap)]
    broken = [s for s in split_sentences(text) if not any(s in p for p in pieces)]
    print(f"overlap={overlap:2d}: {len(pieces)} chunks, {len(broken)} of {len(split_sentences(text))} sentences not intact anywhere")

**Exercise 2.** Semantic chunking on *pairs* of sentences: embed each sentence joined with the next one (`buffer`), recompute the distances and the 80th-percentile cuts. Do the cuts move?

In [ ]:
# Solution — try it yourself first, then run this
buffered = [" ".join(sents[max(0, i - 1): i + 2]) for i in range(len(sents))]
bv = encode(buffered)
bdist = 1 - np.sum(bv[:-1] * bv[1:], axis=1)
_, bstarts = semantic_breakpoints(bdist, 80)
print("no buffer :", starts)
print("buffer = 1:", bstarts, " (windows overlap, so a boundary shows up as a smeared, shifted peak)")

**Exercise 3.** Implement sentence-window retrieval: find the best child sentence, return it with one sentence on each side from the same parent.

In [ ]:
# Solution — try it yourself first, then run this
def sentence_window(question, window=1):
    best = int(np.argmax(child_vecs @ encode([question])[0]))
    pid = children[best][0]
    same_parent = [i for i, (p, _) in enumerate(children) if p == pid]
    pos = same_parent.index(best)
    keep = same_parent[max(0, pos - window): pos + window + 1]
    return " ".join(children[i][1] for i in keep)


print(sentence_window("Which gateway can be installed outdoors?"))

## Say it in an interview

- **30 seconds:** "A chunk is the unit of retrieval and of context. Whole documents embed as a vague average and get truncated by the embedder's window; tiny chunks match precisely but lose the answer. I start with recursive splitting at a few hundred tokens with 10-20% overlap, switch to heading-aware splitting with the heading path prepended as soon as the corpus has structure, and choose the size by sweeping it against a labelled question set - hit-rate@k on one axis, tokens per prompt on the other."
- **Numbers:** MiniLM reads 256 tokens; typical chunks 200-800 tokens; overlap 10-20%; semantic chunking commonly cuts at the 95th percentile of neighbour distances (tune it).
- **Parent-child:** index sentences or ~150-token children, return ~1k-token parents, dedupe by parent id.
- **Metadata:** source, section path, version, updated_at, tenant, ACL, language, content hash, embedding model id.
- **Traps:** "increase the chunk size" is rarely the fix (it dilutes and truncates); semantic chunking on a structured corpus is paying for what headings give free; forgetting the embedder's max length; not deduplicating parents.
- **Follow-ups:** "how would you chunk tables?" (keep the table whole, repeat the header row on every split, index rows as sentences); "what about code?" (split on the AST: functions and classes).

## Next

- [BM25 from scratch](02_bm25_from_scratch.ipynb) - the lexical lane, built on these section chunks.
- [Vector search with FAISS and HNSW](03_vector_search_faiss_hnsw.ipynb) · [Hybrid search and RRF](04_hybrid_search_and_rrf.ipynb) · [RAG pipeline from scratch](06_rag_pipeline_from_scratch.ipynb)
- [LangChain loaders, splitters, vector stores](../13_langchain/04_loaders_splitters_vectorstores_retrievers.ipynb) · [RAG evaluation](../12_evaluation/03_rag_evaluation_retrieval_and_faithfulness.ipynb)
- Theory: [genai_flow course](../../genai_flow/index.html) (run six splitters live) · [AI system design concepts](../../ai_system_design_concepts/index.html) · [interview bank](../../ai_interview_prep/index.html)